# Bulk-crystal CXR — analysis driver

Thin driver for the Monte-Carlo CXR pipeline; the heavy lifting lives in `src/`:

- **`cxr_sweep.py`** — a parameter `Sweep` (each parameter a single value *or* a
  range) expanded into cases.
- **`cxr_run.py`** — run the cases with checkpointing + per-chunk feedback.
- **`cxr_results.py`** — post-processing, the `best_azimuth` reduction, the stats table.
- **`cxr_plots.py`** — all the figures.
- **`cxr_montecarlo.py` / `timepix_response.py`** — the physics + detector model.

Set everything in the **PARAMETERS** cell, then run top to bottom.

In [ ]:
# TODO: Add plots showing the average lifetimes of electrons vs penetration depth

# TODO: Clean up formatting of current monte carlo electron trajectory plots -
#   The x-y figure size and the scaling is inconsistent, as is the x-y location
#   of the slab on the figure, the green 'to detector' arrow and text often
#   aren't in the right spot or aren't pointing the right way, and the 'beam'
#   text sometimes ends up off-figure. Also, the electron trajectories are
#   kinda hard to see, especially as they get towards the end. Maybe slightly
#   increase line thickness. Ensure the figure scaling from plot to plot for the
#   same material thickness is the same, so that only the slab is rotated in frame,
#   with no other scaling/limit/translation changes.
#
#   Also, for each electron energy, put all electron trajectories together
#   on the same figure as with the heatmaps for each combination of
#   polar/azimuthal anglues

# TODO: Remove detector-convolved plots shown in the browser after the sweep
#    replace them with plots which implement the EagleXO detector model

# TODO: Potentially change from matplotlib to another plotting library
#   if any such libraries exist which are much less computationally
#   heavy/are faster, since these matplotlib plots are huge and very slow

# TODO: Split up this notebook into a Jupyter notebook data visualization
#   and a separate scan-runner notebook/script


import sys

sys.path.insert(0, "src")

# Interactive click-through viewer (browse) needs the ipympl backend. For a clean
# PDF export (export_pdf.py), switch this to `%matplotlib inline` -- browse() then
# falls back to drawing every tilt stacked, so all the figures still render.
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from cxr_sweep import Sweep, build_cases, geometry_table
from cxr_results import Settings, records, best_azimuth, show_summary, filter_results
from cxr_plots import (
    browse,
    plot_by_energy,
    plot_peak_vs_tilt,
    stream_chunk,
    plot_timepix_efficiency,
    plot_timepix_detected,
    plot_timepix_poisson,
    plot_heatmaps,
    plot_full_spectrum,
    plot_electron_trajectories,
)
from cxr_run import run_sweep

In [ ]:
# ============================================================================
#  PARAMETERS -- each physical quantity is a SINGLE VALUE (fixed) or a
#  LIST/ARRAY (swept). build_cases takes the Cartesian product over the swept
#  ones; nothing else needs to change to switch a parameter between fixed and
#  swept.
# ============================================================================

settings = Settings(
    beam_current_na=5.0,
    n_electrons=600,  # transport electrons per line spectrum
    n_electrons_brem=150,  # transport electrons per background
    apply_detector_qe=True,
    convolve_with_det=False,
    brem_source="mc",  # "mc" | "external" | "none"
)

# When tilt_azim_deg is swept, collapse it: for each (polar tilt, energy) keep
# only the azimuth with the highest spectral peak max(spectrum). False -> show
# every azimuth.
COLLAPSE_AZIMUTH = True

# Material Choices:
# "hopg" | "diamond" | "silicon"
# "mose2" | "wse2" | "hfse2" | "ws2" | "mos2" | "zrse2" | "ptse2"
MATERIAL = "ptse2"

match MATERIAL:
    case "hopg":
        THICKNESS_ANG = 17e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 40, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-89, -0.1, 20, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 30.0)

    case "diamond":
        THICKNESS_ANG = 10e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89.9, 89.9, 60, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-89.9, -0.1, 30, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 30.0)

    case "silicon":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 40, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-89, -0.1, 15, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = (np.arange(50.0, 4500.0, 1.0),)
        E_GRID_BREM = np.arange(0.0, 60000.0, 30.0)

    case "mose2":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 40, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-89, -0.1, 15, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 30.0)

    case "wse2":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 30, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-85, -0.1, 15)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 25.0)

    case "ptse2":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 40, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-89, -0.1, 15, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 25.0)

    case "hfse2":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 40, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-89, -0.1, 15, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 30.0)

    case "zrse2":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 50, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-89, -0.1, 15, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 25.0)

    case "ws2":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 30, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-85, -0.1, 15, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 3500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 25.0)

    case "mos2":
        THICKNESS_ANG = 1e4
        ELECTRON_ENERGY_KEV = [30, 45, 60]
        TILT_DEG = np.linspace(-89, 89, 60, endpoint=True)
        TILT_AZIM_DEG = np.linspace(-85, -0.1, 15, endpoint=True)
        THETA_OBS_DEG = 90.0
        E_GRID_LINE = np.arange(50.0, 4500.0, 1.0)
        E_GRID_BREM = np.arange(0.0, 60000.0, 25.0)

sweep = Sweep(
    material=MATERIAL,
    thickness_ang=THICKNESS_ANG,
    energy_keV=ELECTRON_ENERGY_KEV,
    tilt_deg=TILT_DEG,
    tilt_azim_deg=TILT_AZIM_DEG,
    theta_obs_deg=THETA_OBS_DEG,  # detector polar angle (fixed)
    E_grid_line=E_GRID_LINE,  # type:ignore
    E_grid_brem=E_GRID_BREM,  # brem: coarse + WIDE (or omit -> auto to 60 keV)
)

In [ ]:
# build the case list and preview the geometry
cases = build_cases(sweep, settings.n_electrons, settings.n_electrons_brem)
print(f"{len(cases)} cases across {len({c['name'] for c in cases})} configs")
display(geometry_table(cases))

In [ ]:
results = {}

try:
    run_sweep(
        cases,
        results,
        on_chunk=lambda batch: stream_chunk(
            results, batch, settings, collapse_azimuth=COLLAPSE_AZIMUTH
        ),
    )
except EOFError:
    print("EOF Error. I think this means the script has already processed all data")

In [ ]:
# Photon-counting stats. best_azimuth collapses the azimuth sweep to one row per
# (polar tilt, energy); set COLLAPSE_AZIMUTH=False above for every azimuth.
res = filter_results(results, cases)  # dict, current sweep only

# The streamed tables print live above; click through the 2x2 best-azimuth plots
# (one figure per polar tilt) here once the run is done.
browse(res, settings, kind="chunk")

plot_heatmaps(
    res, settings, cases=cases, line_metric="prominence"
)  # dominant line -> smoother maps

recs = best_azimuth(records(res)) if COLLAPSE_AZIMUTH else records(res)
show_summary(recs, settings)  # list in

## Timepix3 detector response

Push the intrinsic spectra through a forward model of the actual 2×2 Timepix3
(Si sensor): photoabsorption → charge sharing → per-pixel threshold counting →
ToT energy noise → Poisson statistics (`timepix_response.py`). The ~1.9 keV
counting threshold sits above much of the MoSe₂ signal, so expect strong sub-2
keV suppression and a charge-loss low-energy tail.

> **⚠ Set `TPX_THICKNESS_UM` / `TPX_BIAS_V` to the real quad values** —
> σ_diff ∝ 1/√(bias) is the single most sensitive knob.

In [ ]:
# Timepix3 forward-model figures. Hardware is a PLACEHOLDER -- set to the real
# quad values.
TPX_THICKNESS_UM = 300.0  ### FILL IN -- Si sensor thickness [um]
TPX_BIAS_V = 100.0  ### FILL IN -- applied bias [V]
res = filter_results(results, cases)  # current sweep only

# single-figure diagnostics (left as-is -- not many of these)
plot_timepix_efficiency(thickness_um=TPX_THICKNESS_UM, bias_v=TPX_BIAS_V)
plot_timepix_poisson(
    res,
    settings,
    integration_s=600.0,
    thickness_um=TPX_THICKNESS_UM,
    bias_v=TPX_BIAS_V,
)

# Click-through viewers -- one interactive figure each, paged by polar tilt.
# (Under %matplotlib inline these fall back to every tilt stacked, for the PDF.)
browse(
    res,
    settings,
    kind="timepix",  # Timepix detected (solid) vs incident (dotted), log
    thickness_um=TPX_THICKNESS_UM,
    bias_v=TPX_BIAS_V,
)
browse(res, settings, kind="by_energy")  # intrinsic | detector-convolved line spectra
browse(res, settings, kind="full")  # full measured range out to the beam energy, log

## Electron Trajectories

In [ ]:
sweep = Sweep(
    material=MATERIAL,
    thickness_ang=THICKNESS_ANG,
    energy_keV=[30, 60],
    tilt_deg=np.linspace(-85, 85, 15),
    tilt_azim_deg=0,
    theta_obs_deg=90.0,  # detector polar angle (fixed)
    E_grid_line=E_GRID_LINE,
    E_grid_brem=E_GRID_BREM,  # brem: coarse + WIDE (or omit -> auto to 60 keV)
)
# build the case list and preview the geometry
cases = build_cases(sweep, settings.n_electrons, settings.n_electrons_brem)

for case in cases:
    plot_electron_trajectories(case)

plt.show()